# 02 — Modeling

Train and evaluate churn models using the shared SQLite feature store.

Outputs:
- Trained model pipeline: `models/xgb_model.pkl`
- Printed metrics: classification report + ROC-AUC + PR-AUC
- Optional PR curve visualization

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
# Make sure DB exists
from src.data_processing import main as build_db

build_db()

In [ ]:
# Train + save model pipeline (reuses project code)
from src.model import train_xgb

artifacts = train_xgb()
len(artifacts.feature_names_out)

In [ ]:
# Optional: Precision–Recall curve
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.model_selection import train_test_split

DB_PATH = PROJECT_ROOT / 'data' / 'processed' / 'churn_features.db'
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql('SELECT * FROM churn_features', conn)

y = df['Churn'].astype(int)
X = df.drop(columns=['Churn'])
if 'customerID' in X.columns:
    X = X.drop(columns=['customerID'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe = artifacts.pipeline
proba = pipe.predict_proba(X_test)[:, 1]
prec, rec, thr = precision_recall_curve(y_test, proba)
ap = average_precision_score(y_test, proba)

fig, ax = plt.subplots(figsize=(6.2, 4.6))
ax.plot(rec, prec, linewidth=2)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f'Precision–Recall Curve (AP={ap:.3f})', fontweight='bold')
ax.grid(True, alpha=0.25)
plt.show()